# BASTION Paper Figures: Airline Traffic (Figures 1 & 6)

Reproduces airline traffic figures from the BASTION paper.

- **Comparison figure (Section 6.1)**: 4-panel TBATS/MSTL/STR/BASTION comparison on monthly U.S. international passenger data (2016+).
- **Decomposition figure (Appendix D)**: BASTION decomposition — Observed, Trend, Seasonality (period 12), Remainder.

`fit_BASTION` with `Ks=[12]`, `Outlier=True`, `nsave=2000, nburn=5000` (~2–4 min).
R reference `.rds` files are loaded when available.

In [ ]:
import os, sys, warnings
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
OUTPUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
EXTDATA_DIR  = os.path.join(PROJECT_DIR, 'BASTION', 'inst', 'extdata')
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
from pybastion import fit_BASTION
from pybastion.datasets import load_airtraffic as _load_raw

print(f'Project root : {PROJECT_DIR}')

In [ ]:
def load_airtraffic():
    df = _load_raw()
    df['date'] = pd.to_datetime(
        df['Year'].astype(int).astype(str) + '-'
        + df['Month'].astype(int).astype(str).str.zfill(2) + '-01')
    return df.rename(columns={'Int_Pax': 'pax'})[['date', 'pax']].reset_index(drop=True)

air = load_airtraffic()
print(f'Airline data : {air.shape[0]} monthly observations')
print(f'  Date range : {air["date"].min().date()} to {air["date"].max().date()}')
print(f'  Pax range  : {air["pax"].min():,} to {air["pax"].max():,}')

In [ ]:
print('Fitting BASTION (nsave=2000, nburn=5000, nchains=2) ... (~2-4 min)')
result = fit_BASTION(
    air['pax'].values,
    Ks=[12],
    Outlier=True,
    cl=0.95,
    obsSV='const',
    nsave=2000,
    nburn=5000,
    nchains=2,
    seed=40,
)
summary = result['summary']
print('Done. Summary keys:', list(summary.keys()))

## Comparison Figure (Section 6.1)

4-panel comparison on 2016+ data: observed points, Trend+Seasonality (black), Trend (red), 95% CI (grey).  R reference files loaded when available.

In [ ]:
mask       = air['date'] >= '2016-01-01'
dates_sub  = air['date'].values[mask]
pax_sub    = air['pax'].values[mask]
idx        = np.where(mask)[0]

trend_mean  = np.asarray(summary['Trend_sum']['Mean']).ravel()[idx]
signal_mean = np.asarray(summary['Signal_sum']['Mean']).ravel()[idx]
signal_lo   = np.asarray(summary['Signal_sum']['CR_lower']).ravel()[idx]
signal_hi   = np.asarray(summary['Signal_sum']['CR_upper']).ravel()[idx]

has_tbats = has_mstl = has_str = False
tbats_sig = tbats_tr = mstl_sig = mstl_tr = None
str_sig   = str_tr = str_sig_lo = str_sig_hi = None

try:
    import rdata

    def _ts(obj, attrs):
        data, dim = np.array(obj), attrs.get('dim', None)
        cols = (attrs.get('dimnames') or [None, None])[1]
        if dim is not None and len(dim) == 2:
            data = data.reshape(dim, order='F')
            return pd.DataFrame(data, columns=list(cols) if cols else None)
        return pd.Series(data)

    _cd    = {**rdata.conversion.DEFAULT_CLASS_MAP, 'ts': _ts}
    _parse = lambda f: rdata.conversion.convert(
        rdata.parser.parse_file(os.path.join(EXTDATA_DIR, f)), constructor_dict=_cd)
    _praw  = lambda f: rdata.conversion.convert(
        rdata.parser.parse_file(os.path.join(EXTDATA_DIR, f)))

    tb = _parse('TBATScomp_airtraffic.rds')
    tbats_tr  = np.asarray(tb['level']).ravel()[idx]
    tbats_sig = tbats_tr + np.asarray(tb['season1']).ravel()[idx]
    has_tbats = True

    ms = _parse('MSTL_airtraffic.rds')
    mstl_tr  = np.asarray(ms['Trend']).ravel()[idx]
    mstl_sig = mstl_tr + np.asarray(ms['Seasonal12']).ravel()[idx]
    has_mstl = True

    sp = _praw('STR_airtraffic.rds')['output']['predictors']
    str_tr     = np.asarray(sp[0]['data']).ravel()[idx]
    str_sig    = str_tr + np.asarray(sp[1]['data']).ravel()[idx]
    str_sig_lo = (np.asarray(sp[0]['lower']) + np.asarray(sp[1]['lower'])).ravel()[idx]
    str_sig_hi = (np.asarray(sp[0]['upper']) + np.asarray(sp[1]['upper'])).ravel()[idx]
    has_str    = True
    print('R reference outputs loaded.')
except Exception as e:
    warnings.warn(f'R references unavailable ({e}). Comparison panels will be blank.')

panels = [
    ('TBATS',   tbats_sig,  None,       None,       tbats_tr,  has_tbats),
    ('MSTL',    mstl_sig,   None,       None,       mstl_tr,   has_mstl),
    ('STR',     str_sig,    str_sig_lo, str_sig_hi, str_tr,    has_str),
    ('BASTION', signal_mean, signal_lo, signal_hi,  trend_mean, True),
]

fig, axes = plt.subplots(2, 2, figsize=(9, 5), sharex=True, sharey=True)
for ax, (name, sig, lo, hi, tr, ok) in zip(axes.flatten(), panels):
    ax.scatter(dates_sub, pax_sub, s=8, color='black', zorder=1)
    if ok and sig is not None:
        if lo is not None:
            ax.fill_between(dates_sub, lo, hi, color='grey', alpha=0.4, zorder=2)
        ax.plot(dates_sub, sig, linewidth=1.5, color='black', zorder=3)
        ax.plot(dates_sub, tr,  linewidth=1.5, color='red',   zorder=4)
    ax.set_title(name + ('' if ok else ' (unavailable)'), fontsize=12)
    ax.set_ylabel('Passengers (M)', fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'figure_airtraffic_comparison.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure_airtraffic_comparison.pdf')

## Decomposition Figure (Appendix D)

Four stacked panels: Observed Data, Trend, Seasonality (period 12), Remainder.  Grey bands show 95% credible intervals.

In [ ]:
dates = air['date'].values
pax   = air['pax'].values

trend_mean = np.asarray(summary['Trend_sum']['Mean']).ravel()
trend_lo   = np.asarray(summary['Trend_sum']['CR_lower']).ravel()
trend_hi   = np.asarray(summary['Trend_sum']['CR_upper']).ravel()
s12_mean   = np.asarray(summary['Seasonal12_sum']['Mean']).ravel()
s12_lo     = np.asarray(summary['Seasonal12_sum']['CR_lower']).ravel()
s12_hi     = np.asarray(summary['Seasonal12_sum']['CR_upper']).ravel()
remainder  = pax - trend_mean - s12_mean

fig, axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True)

ax = axes[0]
ax.scatter(dates, pax, s=8, color='black', zorder=2)
ax.plot(dates, trend_mean + s12_mean, linewidth=1.5, color='black', zorder=3)
ax.plot(dates, trend_mean,            linewidth=1.5, color='red',   zorder=4)
ax.set_title('Observed Data', fontsize=12)
ax.set_ylabel('Passengers (M)', fontsize=10)

ax = axes[1]
ax.fill_between(dates, trend_lo, trend_hi, color='grey', alpha=0.5)
ax.plot(dates, trend_mean, linewidth=1.5, color='black')
ax.set_title('Trend', fontsize=12)
ax.set_ylabel('Passengers (M)', fontsize=10)

ax = axes[2]
ax.fill_between(dates, s12_lo, s12_hi, color='grey', alpha=0.5)
ax.plot(dates, s12_mean, linewidth=1.5, color='black')
ax.set_title('Seasonality (period 12)', fontsize=12)
ax.set_ylabel('Passengers (M)', fontsize=10)

ax = axes[3]
ax.plot(dates, remainder, linewidth=1.0, color='black')
ax.axhline(0, color='grey', linestyle='--', linewidth=0.5)
ax.set_title('Remainder', fontsize=12)
ax.set_ylabel('Passengers (M)', fontsize=10)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=8)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'figure_airtraffic_decomposition.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure_airtraffic_decomposition.pdf')